# nb06 - Spark 4.x feature lab

Everything new in 4.0-4.2 that we have not already used, applied to *our* data:

| feature | since | where |
|---------|-------|-------|
| **VARIANT** type (`parse_json`, `variant_get`) | 4.0 / GA 4.1 | semi-structured `vendor_meta` |
| **Geospatial** `GEOMETRY`/`GEOGRAPHY` + `ST_*` | 4.2 | country centroids, distances |
| **Arrow-optimized Python UDFs** (default ON) | 4.2 | scoring UDF with the new UI metric |
| **SQL `CHANGES` clause** (CDC) | 4.2 | reading Delta's change feed in SQL |
| **QUALIFY / pipe syntax** | 4.0/4.2 | covered in nb01, revisited in the wild |

**Prerequisite:** nb02 (silver orders) and ideally nb04 (CDF versions). Cells that
depend on an optional feature are guarded so the notebook always runs to completion.


In [1]:
from pyspark.sql import functions as F

from de_stack import get_spark, BRONZE, register_table

spark = get_spark("nb06-feature-lab", shuffle_partitions=8)

for db, name in [("silver", "orders"), ("silver", "customers")]:
    if not register_table(spark, db, name):
        raise RuntimeError(f"silver.{name} missing - run nb02 first")
print("ready")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 11:40:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


ready


26/10/13 11:40:44 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


## 1. VARIANT: query JSON *without* fixing the schema

Bronze orders carry `vendor_meta` as an **untyped JSON string** with inconsistent keys
across rows - the classic schema-hell. Since 4.0 (GA in 4.1) Spark has a native
**VARIANT** type: semi-structured data that keeps its shape but stays queryable with
subfields, filters, and even shredding (4.1) for fast Parquet storage.


In [2]:
raw = spark.read.json(f"{BRONZE}/orders_raw").select("order_id", "vendor_meta")

# the old way: from_json needs ONE schema that every row must obey
old = (raw.withColumn("parsed", F.expr("from_json(vendor_meta, 'STRUCT<vendor_id:STRING, warehouse:STRING>')"))
           .select("order_id", "vendor_meta", "parsed.*"))
old.show(3, truncate=80)

# the 4.x way: parse into VARIANT - no schema commitment
v = (raw.withColumn("v", F.expr("parse_json(vendor_meta)"))
         .select("order_id", "v"))
v.printSchema()


+--------+--------------------------------------------------------------------------------+---------+---------+
|order_id|                                                                     vendor_meta|vendor_id|warehouse|
+--------+--------------------------------------------------------------------------------+---------+---------+
| O925833|                                          {"vendor_id": "V23", "partner": "xyz"}|      V23|     NULL|
| O949320|                                          {"vendor_id": "V37", "partner": "xyz"}|      V37|     NULL|
| O940468|{"vendor_id": "V7", "warehouse": "WH-APAC-1", "shipping": {"carrier": "fedex"...|       V7|WH-APAC-1|
+--------+--------------------------------------------------------------------------------+---------+---------+
only showing top 3 rows
root
 |-- order_id: string (nullable = true)
 |-- v: variant (nullable = true)



In [3]:
# subfield access: typed projections straight out of the variant
spark.sql('''
    SELECT order_id,
           v:vendor_id::STRING        AS vendor_id,
           v:warehouse::STRING        AS warehouse,
           v:shipping:carrier::STRING AS carrier,
           v:shipping:eta_days::INT   AS eta_days
    FROM (
        SELECT order_id, parse_json(vendor_meta) AS v
        FROM silver.orders
    )
    WHERE v:warehouse IS NOT NULL
    LIMIT 5
''').show(truncate=False)

# variants work in GROUP BY too - and the inconsistent-key rows simply yield NULL
spark.sql('''
    SELECT v:warehouse::STRING AS warehouse, COUNT(*) AS n
    FROM (SELECT parse_json(vendor_meta) AS v FROM silver.orders)
    GROUP BY 1 ORDER BY n DESC
''').show()


26/10/13 11:40:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+---------+---------+-------+--------+
|order_id|vendor_id|warehouse|carrier|eta_days|
+--------+---------+---------+-------+--------+
|O900444 |V36      |WH-APAC-1|dpd    |11      |
|O900476 |V12      |WH-EU-1  |dhl    |3       |
|O900885 |V32      |WH-APAC-1|dpd    |7       |
|O901349 |V28      |WH-US-2  |dhl    |4       |
|O901517 |V20      |WH-EU-1  |ups    |11      |
+--------+---------+---------+-------+--------+



+---------+-----+
|warehouse|    n|
+---------+-----+
|WH-APAC-1|17744|
|  WH-US-2|17574|
|  WH-EU-1|17549|
|     NULL| 5980|
+---------+-----+



> `from_json` needs a schema up front and mangles rows that deviate from it; `parse_json`
> commits to nothing and lets each row keep its own shape. That is the whole pitch.


## 2. Geospatial types (4.2): `GEOMETRY`, `GEOGRAPHY`, SRIDs

Native spatial types land in 4.2: values live as **WKB** (Well-Known Binary) with an
**SRID** from a built-in PROJ-based registry. The *initial* function set is deliberately
small - construct, inspect, persist (`ST_GeomFromWKB`, `ST_GeogFromWKB`, `ST_AsBinary`,
`ST_Srid`, `ST_SetSrid`); measurement/overlay functions such as `ST_Distance` are on the
SPIP roadmap for later releases. Also note the SQL syntax rule: columns must be declared
with an explicit SRID - `GEOGRAPHY(4326)` or `GEOMETRY(ANY)`, never bare `GEOMETRY`.

We encode country centroids as WKB (a point is just: byte order + type + lon + lat).


In [4]:
import struct

def wkb_point(lon: float, lat: float) -> bytes:
    return struct.pack("<BIdd", 1, 1, lon, lat)   # NDR, type=Point, x, y

centroids = [("DE", 10.4515, 51.1657), ("FR", 2.2137, 46.2276),
             ("GB", -3.4360, 55.3781), ("US", -95.7129, 37.0902),
             ("IN", 78.9629, 20.5937)]

(spark.createDataFrame([(c, wkb_point(lon, lat)) for c, lon, lat in centroids],
                       "country STRING, wkb BINARY")
     .createOrReplaceTempView("geo_points"))

spark.sql('''
    SELECT country,
           ST_Srid(ST_GeogFromWKB(wkb))            AS geog_srid,   -- always 4326
           hex(ST_AsBinary(ST_GeogFromWKB(wkb)))   AS wkb_roundtrip
    FROM geo_points
''').show(truncate=False)


+-------+---------+------------------------------------------+
|country|geog_srid|wkb_roundtrip                             |
+-------+---------+------------------------------------------+
|DE     |4326     |0101000000BA490C022BE724403E7958A835954940|
|FR     |4326     |0101000000D93D7958A8B50140E9482EFF211D4740|
|GB     |4326     |0101000000B0726891ED7C0BC00E4FAF9465B04B40|
|US     |4326     |010100000061545227A0ED57C0B30C71AC8B8B4240|
|IN     |4326     |010100000061545227A0BD53409CA223B9FC973440|
+-------+---------+------------------------------------------+



**SRID validation** is enforced by the registry - try to invent a coordinate system or
give `GEOGRAPHY` a projected one, and the query refuses:


In [5]:
print("projected SRID on GEOMETRY:",
      spark.sql("SELECT ST_Srid(ST_SetSrid(ST_GeomFromWKB(wkb), 3857)) FROM geo_points LIMIT 1").first()[0])

try:
    spark.sql("SELECT ST_GeomFromWKB(wkb, 99999) FROM geo_points LIMIT 1").collect()
except Exception as e:
    print("invented SRID rejected:", str(e).splitlines()[0][:110])

try:
    spark.sql("CREATE TABLE bad_geo (id INT, loc GEOGRAPHY(3857))")  # 3857 is projected
except Exception as e:
    print("projected SRID on GEOGRAPHY rejected:", str(e).splitlines()[0][:110])


projected SRID on GEOMETRY: 3857


invented SRID rejected: [ST_INVALID_SRID_VALUE] Invalid or unsupported SRID (spatial reference identifier) value: 99999. SQLSTATE: 220
projected SRID on GEOGRAPHY rejected: [ST_INVALID_SRID_VALUE] Invalid or unsupported SRID (spatial reference identifier) value: 3857. SQLSTATE: 2202


**Persistence**: Delta/Parquet store a fixed-SRID column's geometry natively (mixed
`GEOMETRY(ANY)` columns are in-memory only). Round-trip from bytes to typed column to disk:


In [6]:
# Ecosystem check: Delta 4.4.1 does NOT accept geospatial columns yet
# (DELTA_GEOSPATIAL_NOT_SUPPORTED); plain Parquet carries them per the new
# parquet-format geospatial spec. Try Delta first so you see the real error.
try:
    spark.sql("DROP TABLE IF EXISTS geo_points_t")
    spark.sql('''
        CREATE TABLE geo_points_t USING DELTA AS
        SELECT country, ST_GeogFromWKB(wkb) AS pt FROM geo_points
    ''')
    print("persisted as Delta")
except Exception as e:
    print("Delta refused:", str(e).splitlines()[0][:110])

# Parquet carries the column (parquet-format geospatial spec); writing to a path with
# overwrite sidesteps managed-table catalog quirks and stays idempotent
(spark.sql("SELECT country, ST_GeogFromWKB(wkb) AS pt FROM geo_points")
     .write.mode("overwrite").parquet("/data/warehouse/geo_points_parquet"))
print("persisted as Parquet at /data/warehouse/geo_points_parquet")

(spark.read.parquet("/data/warehouse/geo_points_parquet")
     .selectExpr("country", "ST_Srid(pt) AS srid")
     .orderBy("country").show(truncate=False))


26/10/13 11:41:11 ERROR Utils: Aborting task
org.apache.spark.sql.delta.DeltaUnsupportedOperationException: [DELTA_GEOSPATIAL_NOT_SUPPORTED] Geospatial types are not supported in this version of Delta Lake.
	at org.apache.spark.sql.delta.DeltaErrorsBase.geoSpatialNotSupportedException(DeltaErrors.scala:392)
	at org.apache.spark.sql.delta.DeltaErrorsBase.geoSpatialNotSupportedException$(DeltaErrors.scala:389)
	at org.apache.spark.sql.delta.DeltaErrors$.geoSpatialNotSupportedException(DeltaErrors.scala:4275)
	at org.apache.spark.sql.delta.GeoSpatialTableFeature$.metadataRequiresFeatureToBeEnabled(TableFeature.scala:705)
	at org.apache.spark.sql.delta.actions.Protocol$$anonfun$1.applyOrElse(actions.scala:424)
	at org.apache.spark.sql.delta.actions.Protocol$$anonfun$1.applyOrElse(actions.scala:422)
	at scala.collection.Iterator$$anon$7.hasNext(Iterator.scala:535)
	at scala.collection.immutable.List.prependedAll(List.scala:152)
	at scala.collection.immutable.List$.from(List.scala:674)
	at s

Delta refused: [DELTA_GEOSPATIAL_NOT_SUPPORTED] Geospatial types are not supported in this version of Delta Lake.


persisted as Parquet at /data/warehouse/geo_points_parquet


+-------+----+
|country|srid|
+-------+----+
|DE     |4326|
|FR     |4326|
|GB     |4326|
|IN     |4326|
|US     |4326|
+-------+----+



## 3. Arrow-optimized Python UDFs (4.2: default ON)

Python UDFs used to ship row-by-row (or pandas-UDF-per-batch with explicit imports).
Since 4.2 the Arrow path - batched, zero-copy, vectorized - is the **default**, and the
UI shows a new *processing time* metric per UDF (SPARK-55046). Compare a scalar Python
UDF against built-in SQL on the same data.


In [7]:
spark.conf.get("spark.sql.execution.arrow.pyspark.enabled")


'true'

In [8]:
import math
import time

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

@udf(DoubleType())
def risk_score(amount):
    # deliberately pythonic: log-scale amount scoring, capped at 10
    if amount is None:
        return None
    return round(min(math.log10(amount + 1) * 2, 10.0), 3)

orders = spark.table("silver.orders").select("total_amount").filter("total_amount IS NOT NULL")

t0 = time.perf_counter()
with_udf = orders.select(risk_score("total_amount").alias("s")).agg(F.avg("s")).first()[0]
t1 = time.perf_counter()

# same math as built-in SQL for reference
t2 = time.perf_counter()
with_sql = (orders.select(F.round(F.least(F.log10(F.col("total_amount") + 1) * 2, F.lit(10.0)), 3).alias("s"))
            .agg(F.avg("s")).first()[0])
t3 = time.perf_counter()

print(f"python UDF : {t1 - t0:5.2f}s   avg score {with_udf:.4f}")
print(f"builtin SQL: {t3 - t2:5.2f}s   avg score {with_sql:.4f}")
print("Driver UI -> SQL tab now shows a per-UDF processing-time metric (new in 4.2).")


python UDF :  3.37s   avg score 3.2695
builtin SQL:  2.03s   avg score 3.2695
Driver UI -> SQL tab now shows a per-UDF processing-time metric (new in 4.2).


## 4. SQL `CHANGES` clause (4.2): CDC without a custom reader

Delta records row-level changes (nb04 read them with `readChangeFeed`). 4.2 generalizes
this with a **`CHANGES` clause** and `changes()` DataFrame API on any DSv2 table that
supports the CDC connector API. If the installed Delta version does not expose that API
yet, the cell degrades gracefully to the classic CDF read - same data, different syntax.


In [9]:
import os

if not os.path.isdir("/data/warehouse/silver.db/orders/_delta_log"):
    raise RuntimeError("silver.orders missing - run nb02 first")

first_version = 1
try:
    changes = spark.sql(f'''
        SELECT _change_type, order_id, total_amount
        FROM silver.orders CHANGES VERSION AS OF {first_version}
        LIMIT 5
    ''')
    changes.show(truncate=False)
    print("SQL CHANGES clause works against this Delta version")
except Exception as e:
    msg = str(e).splitlines()[0]
    print(f"SQL CHANGES not supported by this connector -> classic CDF fallback")
    print("reason:", msg[:140])
    (spark.read.format("delta")
          .option("readChangeFeed", "true")
          .option("startingVersion", first_version)
          .table("silver.orders")
          .select("_change_type", "order_id", "total_amount")
          .show(5, truncate=False))


SQL CHANGES not supported by this connector -> classic CDF fallback
reason: 


+----------------+--------+------------+
|_change_type    |order_id|total_amount|
+----------------+--------+------------+
|update_preimage |O900135 |81.74       |
|update_postimage|O900135 |81.74       |
|update_preimage |O900519 |54.21       |
|update_postimage|O900519 |54.21       |
|update_preimage |O900569 |174.06      |
+----------------+--------+------------+
only showing top 5 rows


## 5. Your turn

- Try `SELECT * FROM silver.orders CHANGES ...` against other versions from nb04's history.
- Check the Spark 4.2 UI (History Server): search the SQL tab for `risk_score` and find
  the new UDF processing-time metric.
- Read the release notes linked in the README and port one more feature into your own query.

Next: **nb07_gold_and_serving.ipynb** - build the gold layer the dashboard reads.
